# Phase 0: Reproduce GCR Baseline

**Goal:** Reproduce GCR's Hits@1 >= 91% on WebQSP.

**Requires:** A100 GPU (40GB VRAM) for the 8B LLM at full precision.
With 4-bit quantization a T4/V100 may work.

**Pipeline:**
1. Step 1: `predict_paths_and_answers.py` — constrained decoding with KG-Trie
2. Step 2: `predict_final_answer.py` — answer extraction (optional)

## 1. Colab / Local Environment Setup

In [ ]:
import sys, os, json, warnings, gc, subprocess
import numpy as np
import torch

IN_COLAB = 'google.colab' in sys.modules
print(f"Python: {sys.version}")
print(f"Running in Colab: {IN_COLAB}")

# Check GPU
cuda_ok = torch.cuda.is_available()
if cuda_ok:
    vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"GPU: {torch.cuda.get_device_name(0)}  VRAM: {vram_gb:.1f} GB")
    if vram_gb < 20:
        print("WARNING: <20GB VRAM. Use 4-bit quantization or reduce num_beams.")
else:
    print("WARNING: No GPU detected. This pipeline requires a GPU for the 8B model.")

if IN_COLAB:
    if not os.path.exists('dca-trie'):
        !git clone https://github.com/YOUR_USERNAME/dca-trie.git
        %cd dca-trie
        !git submodule update --init
    else:
        %cd dca-trie
    !pip install -q -e .
else:
    print("Running locally. Ensure poetry env is active.")

## 2. Set HuggingFace Token

The GCR model is gated.
Token: https://huggingface.co/settings/tokens
Access: https://huggingface.co/rmanluo/GCR-Meta-Llama-3.1-8B-Instruct

In [ ]:
from huggingface_hub import login

HF_TOKEN = "" or os.environ.get("HF_TOKEN", "")
if HF_TOKEN:
    login(token=HF_TOKEN)
    print("HF_TOKEN configured.")
else:
    raise ValueError("HF_TOKEN required. Set it above or in .env")

## 3. Step 1: Run GCR Path Prediction

Constrained decoding with KG-Trie on 100 questions.
Takes ~45 min on A100.

**Flags you may want to tune:**
- `--n 100`: number of questions (use 10 for a quick smoke test first)
- `--model_name gcr-Llama-2-7b-chat-hf`: GCR's original smaller model
- `--attn_implementation flash_attention_2`: requires flash-attn installed
- `--load_in_4bit`: use 4-bit quantization (needs less VRAM)

In [ ]:
%%time
!python gcr/workflow/predict_paths_and_answers.py \
    --model_name rmanluo/GCR-Meta-Llama-3.1-8B-Instruct \
    --model_path rmanluo/GCR-Meta-Llama-3.1-8B-Instruct \
    --data_path rmanluo \
    --d RoG-webqsp \
    --split test[:100] \
    --predict_path results/GenPaths \
    --n 1 \
    --max_new_tokens 128 \
    --k 10 \
    --generation_mode beam \
    --index_path_length 2 \
    --dtype bf16 \
    --attn_implementation sdpa

print("Step 1 complete.")

## 4. Check Results

In [ ]:
import glob

pred_files = glob.glob("results/GenPaths/RoG-webqsp/rmanluo/GCR-Meta-Llama-3.1-8B-Instruct/*/predictions.jsonl")
print(f"Found {len(pred_files)} prediction files")

if pred_files:
    import pandas as pd
    df = pd.read_json(pred_files[0], lines=True)
    print(f"\nPredictions: {len(df)}")
    print(f"Columns: {list(df.columns)}")
    print(f"\nSample:")
    print(df.iloc[0].to_dict() if len(df) > 0 else "empty")

## 5. Evaluate Hits@1 and F1

In [ ]:
if pred_files:
    reasoning_path = pred_files[0]
    print(f"Reasoning paths: {reasoning_path}")
    !python gcr/workflow/predict_final_answer.py \
        --data_path rmanluo \
        --d RoG-webqsp \
        --split test[:100] \
        --predict_path results/KGQA \
        --add_path true \
        --reasoning_path {reasoning_path} \
        --model_name rmanluo/GCR-Meta-Llama-3.1-8B-Instruct \
        --model_path rmanluo/GCR-Meta-Llama-3.1-8B-Instruct \
        --attn_implementation sdpa \
        --k 10 \
        --generation_mode beam \
        --dtype bf16
else:
    print("No prediction files found. Run Step 1 first.")

## Phase 0 Exit Criteria
- [ ] Step 1 runs without errors
- [ ] Hits@1 >= 91% on 100 questions
- [ ] Predictions saved to `results/GenPaths/.../predictions.jsonl`